## _SCRAPPING_ DE VÍDEOS DE YOUTUBE
---

Para extraer vídeos de Youtube de forma eficaz usaremos el siguiente proceso:

- **YouTube tiene una API oficial** que permite usar ciertas funcionalidades, entre ellas hacer busquedas como un usuario haría en la plataforma y obtener información superficial de los resultados.

La idea es usar esa API, por dos razones:

1. No tenemos que temer el bloqueo de IP, como es el método oficial, no nos va a ocurrir que nos bloquen, y no vamos a tener que meter un _delay_ artificial entre peticiones.

2. En la búsqueda, podemos tener información del título y el ID del vídeo, la mayoría de librerías que se ha visto permiten _scrappear_ pero se necesita la ID del vídeo del cual queremos la información.

**PROBLEMA:** la API oficial no es del todo gratis, hacer peticiones como busquedas consume un saldo. Google nos da gratuitamente, $10.000$ unidades (o monedas) para gastar por día, y se renuevan todos los días. Hacer una busqueda gasta $100$ unidades, y en cada una de las busquedas se pueden extraer $50$ noticias, lo cuál nos deja un techo de $100$ busquedas o la posibilidad de _scrappear_ $5000$ vídeos en un día.

Al ser un techo de uso bastante alto, no hay mucho problema.

Para poder usar la API hay que crear una _key_. En [este](https://www.youtube.com/watch?v=WEOgqSx4ZZw) vídeo enseña cómo en los primeros minutos.

In [ ]:
# AQUÍ CREAMOS Y CONFIGURAMOS EL OBJETO DE PYTHON QUE NOS SERVIRÁ PARA HACER PETICIONES A LA API DE YOUTUBE
from googleapiclient.discovery import build

# La key
API_KEY = # CREARSE UNA KEY EN GOOGLE

# El objeto
youtube = build('youtube', 'v3', developerKey=API_KEY)

In [52]:
# Hacer las peticiones a la API se hace en dos pasos:

# Primero se configura los parámetros de la búsqueda
busqueda = youtube.search().list(
    part="snippet",
    q="inteligencia artificial",    # Esto es la búsqueda que hará, será lo que se escriba en la barra del buscador
    type="video",
    maxResults=50,           # máximo por llamada, paginas con nextPageToken
    videoDuration="medium",  # filtro para la duración, short <4min, medium 4-20min, long >20min
    order="relevance",       # criterio para ordenar los elementos de la búsqueda, puede ser "viewCount", "date" o el que hay
    relevanceLanguage="es",
    publishedAfter="2024-01-01T00:00:00Z"
)


In [49]:
# Después se ejecuta
# La línea que hace la petición a la API es esta, es la que hace que se gaste el saldo que tenemos en la key, así que cuidado
# con lo de ejecutarla varias veces
response = busqueda.execute()

In [53]:
# Esto es para ver lo que nos ha devuelto la consulta
# También ayuda a saber cómo funciona un poco mejor por dentro el objeto response que devuelve la búsqueda
for item in response['items']:
    titulo = item['snippet']['title']
    video_id = item['id']['videoId']
    canal = item['snippet']['channelTitle']
    fecha = item['snippet']['publishedAt']
    
    print(f"Título: {titulo}")
    print(f"URL: https://www.youtube.com/watch?v={video_id}")
    print(f"ID video: {video_id}")
    print(f"Canal: {canal}")
    print(f"Fecha: {fecha}")
    print("---")

Título: 🤖Curso BÁSICO de Inteligencia Artificial - Curso Google (Resumen en 10 minutos)
URL: https://www.youtube.com/watch?v=2bnViboSm8A
ID video: 2bnViboSm8A
Canal: Yovany con Y
Fecha: 2024-06-17T14:00:01Z
---
Título: El juego de tronos de la IA ya es una realidad
URL: https://www.youtube.com/watch?v=JrSKJLSab-U
ID video: JrSKJLSab-U
Canal: MoureDev by Brais Moure
Fecha: 2026-05-12T14:00:34Z
---
Título: Conecto la nueva voz de ChatGPT a mis Agentes: ahora hablan de verdad
URL: https://www.youtube.com/watch?v=RUDBhauy7d4
ID video: RUDBhauy7d4
Canal: Inteligencia Artificial
Fecha: 2026-05-12T18:00:06Z
---
Título: Ya tenemos al GANADOR de la carrera por la IA (y no es OpenAI)
URL: https://www.youtube.com/watch?v=uRK8hWFkhWY
ID video: uRK8hWFkhWY
Canal: EDteam
Fecha: 2026-05-13T23:25:16Z
---
Título: Curso de IA de Google para principiantes (Resumen en 10 minutos)
URL: https://www.youtube.com/watch?v=-idMBeCCCzs
ID video: -idMBeCCCzs
Canal: DonebyLaura
Fecha: 2024-05-15T17:39:59Z
---
Títul

## TRANSCIPCIÓN
---

Pasamos a la segunda fase: obtener la transcripción de los vídeos. Aquí el modo de trabajar es diferente, la API que usamos para extraer la transcripción no es oficial, de manera que es un poco caja negra, pero el mayor problema es que YouTube puede darnos problemas si hacemos demasiadas peticiones. Lo peor que podría pasar es que detecte nuestras peticiones como un robot y nos devuelva un error.

Para solucionar este problema debemos poner un tiempo de espera entre peticiones, esto va a ralentizar mucho el conseguir la información, pero no hay muchas alternativas.

Vamos a poner que el tiempo de espera sea entre $5$ y $10$ segundos, podríamos hacer pruebas para ver si es posible esperar menos tiempo y agilizar el proceso, pero es preferible ser conservadores en este aspecto, principalmente para evitar un posible bloqueo de la IP o similares.

Documentación: [aquí](https://pypi.org/project/youtube-transcript-api/)

In [54]:
from youtube_transcript_api import YouTubeTranscriptApi

# Vamos a hacer un prueba con un video normal, solo debemos especificar el identificador que YouTube le da
ID_video = "RUDBhauy7d4"

# Según explica la documentación oficial, para obtener la transcripción hacemos:
ytt_api = YouTubeTranscriptApi()
transcripcion = ytt_api.fetch(video_id,
                              languages=['es'])         # Idiomas de las transcripcion

# El objeto que devuelve el método fetch es algo extraño, se parece mucho a una lista y a un diccionario, básicamente
# contiene las frases que se dicen en el video y junto con el texto las marcas de tiempo de cuando empieza y termina de 
# decir esa frase, es decir, contiene TODA la transcripción que YouTube muestra en su página web

# Para poder trabajar con este objeto para extraer el texto completo hacemos lo siguiente:
transcripcion = transcripcion.to_raw_data()
# Esto convierte el objeto anterior en una lista con diccionarios, para más detalles consultar la documentación donde hay 
# una explicación gráfica, pero básicamente esto nos permite usar lo que ya sabemos de listas

texto = " ".join([marca['text'] for marca in transcripcion])
texto

IpBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=xec6rDvnly4! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide th

## ENSAMBLADO Y APLICACIÓN:
---

Vamos a juntar todo lo anterior para crear una función que haga la busqueda, tome todos los resultados, consiga la transcripción y lo guarde todo en un JSON.

In [55]:
# Librerías
import time
import random

In [56]:
ytt_api = YouTubeTranscriptApi()
def transcribir(ID):
    # El try y el except es porque existen vídeos raros que no tienen transcripción o lo que sea, y que al intentar obtenerla
    # se devuelve un error por parte de la aplicación que usamos error, así evitamos perder mucho trabajo, pues la función 
    # principal va a tardar mucho en ejecutar
    try:
        transcripcion = ytt_api.fetch(ID, languages=['es'])
        return " ".join([m['text'] for m in transcripcion.to_raw_data()])
    except Exception:
        return None  # o "" si prefieres string vacío

In [57]:

def scrapeando_youtube(consulta, duracion, num_videos=50, idioma="es", a=5, b=10):
    
    busqueda = youtube.search().list(part="snippet",
                                     q=consulta,
                                     type="video",
                                     maxResults=num_videos,
                                     videoDuration=duracion,
                                     order="relevance",
                                     relevanceLanguage=idioma,
                                     publishedAfter="2024-01-01T00:00:00Z")
    response = busqueda.execute()
    print("Finalizada la búsqueda")
    videos = []
    i = 1
    for item in response['items']:

        video = {
            "Título": item['snippet']['title'],
            "ID": item['id']['videoId'],
            "Canal": item['snippet']['channelTitle'],
            "Fecha": item['snippet']['publishedAt'],
            "Transcripcion": transcribir(item['id']['videoId'])}
        
        videos.append(video)
        print(f"Fin iteración {i}")
        i=+1
        # Tiempo de espera para evitar bloqueos
        time.sleep(random.uniform(a, b))
    
    return videos

In [34]:
videos = scrapeando_youtube("inteligencia artificial", "medium")

Finalizada la búsqueda
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1
Fin iteración 1


In [35]:
videos

[{'Título': 'Conecto la nueva voz de ChatGPT a mis Agentes: ahora hablan de verdad',
  'ID': 'RUDBhauy7d4',
  'Canal': 'Inteligencia Artificial',
  'Fecha': '2026-05-12T18:00:06Z',
  'Transcripcion': None},
 {'Título': 'Ya tenemos al GANADOR de la carrera por la IA (y no es OpenAI)',
  'ID': 'uRK8hWFkhWY',
  'Canal': 'EDteam',
  'Fecha': '2026-05-13T23:25:16Z',
  'Transcripcion': None},
 {'Título': '🤖Curso BÁSICO de Inteligencia Artificial - Curso Google (Resumen en 10 minutos)',
  'ID': '2bnViboSm8A',
  'Canal': 'Yovany con Y',
  'Fecha': '2024-06-17T14:00:01Z',
  'Transcripcion': None},
 {'Título': 'Curso de IA de Google para principiantes (Resumen en 10 minutos)',
  'ID': '-idMBeCCCzs',
  'Canal': 'DonebyLaura',
  'Fecha': '2024-05-15T17:39:59Z',
  'Transcripcion': None},
 {'Título': '“China ya empató a EU en Inteligencia Artificial”: Ruiz-Healy lanza alerta tecnológica',
  'ID': 'KnSzPPNuf_8',
  'Canal': 'Grupo Fórmula',
  'Fecha': '2026-05-14T23:03:03Z',
  'Transcripcion': None},


## PARALELIZANDO
---

Si no tememos al bloqueo de IP por parte de YouTube (cosa que deberíamos hacer), podemos acelerar el proceso de obtención de los datos paralelizando.

La idea es que nuestro ordenador, desde diferentes puertos se conecte a los servidores de YouTube, y haga diferentes peticiones en cada puerto, así se transcriben varios vídeos a la vez.

## GUARDADO DE LOS DATOS
---

In [45]:
import json
import os

# Para guardar los datos obtenidos
def guardar_json(videos, ruta):
    if os.path.exists(ruta):
        with open(ruta, 'r', encoding='utf-8') as f:
            datos_existentes = json.load(f)
        datos_existentes.extend(videos)
        datos_finales = datos_existentes
    else:
        datos_finales = videos

    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(datos_finales, f, ensure_ascii=False, indent=2)
    
    print(f"{len(videos)} vídeos guardados en '{ruta}' (total: {len(datos_finales)})")

In [46]:
guardar_json(videos, "resultados.json")

50 vídeos guardados en 'resultados.json' (total: 50)
